# Mission 6: Advanced Arena Agents — Optimal Market Making and Execution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_06_advanced_agents/notebook.ipynb)

**Learning objectives**
- Understand *why* the naive market maker loses money (adverse selection)
- Implement and tune the Avellaneda-Stoikov (2008) optimal quoting model
- Build and analyze a TWAP execution agent
- Observe mean reversion vs. momentum dynamics in the same microstructure
- Run a head-to-head tournament across all agent types and interpret the results

> **Note:** This mission runs entirely in-process — no Arena server required.  All simulations use the same LOB engine and agent population that power the live Arena.

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q convexpi-lab>=0.1.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from convexpi.arena.market import Market
from convexpi.arena import (
    Agent, MarketState, Order, Side,
    NoiseTrader, NaiveMarketMaker, MomentumTrader, InformedTrader,
    AvellanedaStoikov, TWAPAgent, MeanReversionAgent,
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
print('Ready.')

In [ ]:
def run_sim(agents, n_ticks=1000, seed=42, fundamental_kwargs=None):
    """
    Run an in-process simulation and return per-agent telemetry DataFrames.

    Returns
    -------
    pnl_df     : DataFrame[agent_id → PnL in dollars] per tick
    inv_df     : DataFrame[agent_id → inventory (shares)] per tick
    market_df  : DataFrame of market snapshots (fundamental, spread, price, volume)
    """
    agent_ids  = [a.agent_id for a in agents]
    telem      = {aid: [] for aid in agent_ids}
    market_log = []

    def _collect(m):
        mark = m.engine.last_price or (m.fundamental.value if hasattr(m.fundamental, 'value') else None)
        if mark is None:
            return
        for aid in agent_ids:
            acct = m.accounts.get(aid)
            if acct:
                telem[aid].append({'pnl': acct.value(mark) / 100, 'inv': acct.position})
        if m.snapshots:
            market_log.append(m.snapshots[-1])

    market = Market(agents, n_ticks=n_ticks, seed=seed,
                    fundamental_kwargs=fundamental_kwargs or {})
    for t in range(1, n_ticks + 1):
        market.at_tick(t, _collect)
    market.run()

    pnl_df = pd.DataFrame({aid: [r['pnl'] for r in rows] for aid, rows in telem.items()})
    inv_df = pd.DataFrame({aid: [r['inv'] for r in rows] for aid, rows in telem.items()})
    market_df = pd.DataFrame(market_log)

    lb = market.leaderboard()
    print(f"{'Rank':<5} {'Agent':<20} {'Final PnL':>12}  {'Inventory':>10}")
    print('-' * 52)
    for rank, (aid, pnl, pos) in enumerate(lb, 1):
        print(f"  {rank:<3} {aid:<20} ${pnl:>10.2f}  {pos:>10}")

    return pnl_df, inv_df, market_df, market

print('run_sim() helper ready.')

---
## Part 1: Anatomy of Adverse Selection

Mission 2 showed that the naive market maker earns the spread from noise traders but gets run over by informed traders.  Let's quantify *exactly* how much of the loss comes from each source.

**Setup:** two simulations — one with only noise traders (MM should profit), one with an informed trader added (MM should lose).

In [ ]:
# Simulation A: noise traders only — naive MM should be profitable
print("=== Simulation A: Noise traders only ===")
agents_A = [
    NoiseTrader('nt1', seed=10),
    NoiseTrader('nt2', seed=11),
    NoiseTrader('nt3', seed=12),
    NoiseTrader('nt4', seed=13),
    NaiveMarketMaker('naive_mm', seed=40),
]
pnl_A, inv_A, mkt_A, _ = run_sim(agents_A, n_ticks=1000, seed=42)

In [ ]:
# Simulation B: noise traders + 1 informed trader
print("=== Simulation B: With informed trader ===")
agents_B = [
    NoiseTrader('nt1', seed=10),
    NoiseTrader('nt2', seed=11),
    NoiseTrader('nt3', seed=12),
    InformedTrader('inf1', seed=20),
    NaiveMarketMaker('naive_mm', seed=40),
]
pnl_B, inv_B, mkt_B, _ = run_sim(agents_B, n_ticks=1000, seed=42)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# PnL over time
axes[0, 0].plot(pnl_A['naive_mm'], color='steelblue', label='noise only')
axes[0, 0].plot(pnl_B['naive_mm'], color='coral',     label='+ informed')
axes[0, 0].axhline(0, color='black', lw=0.8)
axes[0, 0].set_title('Naive MM: PnL over time')
axes[0, 0].legend()
axes[0, 0].set_ylabel('PnL ($)')

# Inventory over time
axes[0, 1].plot(inv_A['naive_mm'], color='steelblue', alpha=0.8)
axes[0, 1].plot(inv_B['naive_mm'], color='coral',     alpha=0.8)
axes[0, 1].axhline(0, color='black', lw=0.8)
axes[0, 1].set_title('Naive MM: Inventory')
axes[0, 1].set_ylabel('Shares')

# Mid-price vs fundamental — sim B
if 'best_bid' in mkt_B.columns and 'best_ask' in mkt_B.columns:
    mid = (mkt_B['best_bid'] + mkt_B['best_ask']) / 2
    axes[1, 0].plot(mkt_B['fundamental'] / 100, color='grey',     label='fundamental', lw=0.8)
    axes[1, 0].plot(mid / 100,                  color='steelblue', label='mid',         lw=1)
    axes[1, 0].set_title('Price vs Fundamental (sim B)')
    axes[1, 0].legend()
    axes[1, 0].set_ylabel('Price ($)')

# Adverse selection cost = PnL diff
adv_cost = pnl_A['naive_mm'].values[:len(pnl_B)] - pnl_B['naive_mm'].values
axes[1, 1].fill_between(range(len(adv_cost)), adv_cost, alpha=0.6, color='red')
axes[1, 1].set_title('Adverse Selection Cost (PnL difference)')
axes[1, 1].set_ylabel('PnL loss from informed flow ($)')

plt.tight_layout()
plt.show()

mm_pnl_noise    = pnl_A['naive_mm'].iloc[-1]
mm_pnl_informed = pnl_B['naive_mm'].iloc[-1]
print(f"Naive MM PnL (noise only) :  ${mm_pnl_noise:.2f}")
print(f"Naive MM PnL (+ informed) :  ${mm_pnl_informed:.2f}")
print(f"Adverse selection cost    :  ${mm_pnl_noise - mm_pnl_informed:.2f}")

---
## Part 2: Avellaneda-Stoikov Optimal Market Making

Avellaneda & Stoikov (2008) solved the optimal quoting problem for a market maker with CARA (constant absolute risk aversion) utility.  The key insight: the MM should skew its quotes away from inventory risk.

**Reservation price** (the MM's "true" value given current inventory $q$):
$$r = s - q \cdot \gamma \cdot \sigma^2 \cdot (T - t)$$

where $s$ is the mid price, $\gamma$ is risk aversion, $\sigma^2$ is price variance, and $T - t$ is time remaining.

**Optimal half-spread:**
$$\delta^* = \frac{\gamma \sigma^2 (T-t)}{2} + \frac{2}{\gamma} \ln\left(1 + \frac{\gamma}{\kappa}\right)$$

where $\kappa$ is the order arrival intensity.  Higher $\kappa$ → more frequent orders → tighter spread.

**What this means practically:**
- Long inventory → reservation price < mid → ask closer to mid than bid (cheaper to sell, harder to buy more)
- Spread widens when variance or risk aversion is high
- Spread narrows as time horizon shrinks (less risk to hold inventory near close)

In [ ]:
# Head-to-head: NaiveMarketMaker vs AvellanedaStoikov
print("=== Naive MM vs Avellaneda-Stoikov ===")
agents_C = [
    NoiseTrader('nt1', seed=10),
    NoiseTrader('nt2', seed=11),
    NoiseTrader('nt3', seed=12),
    InformedTrader('inf1', seed=20),
    MomentumTrader('mom1', seed=30),
    NaiveMarketMaker('naive_mm',  seed=40),
    AvellanedaStoikov('as_mm',    seed=50, gamma=0.1, kappa=1.5, size=15, horizon=1000),
]
pnl_C, inv_C, mkt_C, mkt_obj_C = run_sim(agents_C, n_ticks=1000, seed=42)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(pnl_C['naive_mm'], label='NaiveMarketMaker',   color='coral',     lw=1.5)
axes[0].plot(pnl_C['as_mm'],    label='AvellanedaStoikov',  color='steelblue', lw=1.5)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Market Maker Comparison: PnL over 1000 Ticks')
axes[0].set_ylabel('PnL ($)')
axes[0].legend()

axes[1].plot(inv_C['naive_mm'], label='NaiveMarketMaker', color='coral',     alpha=0.7, lw=1)
axes[1].plot(inv_C['as_mm'],    label='AvellanedaStoikov', color='steelblue', alpha=0.7, lw=1)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Inventory over Time')
axes[1].set_ylabel('Shares')
axes[1].legend()

plt.tight_layout()
plt.show()

### Tuning the A-S Parameters

The A-S model has two key parameters:

| Parameter | Effect | Too high | Too low |
|---|---|---|---|
| `gamma` (risk aversion) | Controls inventory penalty | Quotes too aggressively on the offloading side; fills dry up | Ignores inventory; builds dangerous positions |
| `kappa` (order arrival) | Controls spread size | Spread too tight; negative edge | Spread too wide; never fills |

Run the parameter sweep below to find the sweet spot.

In [ ]:
# Parameter sweep: gamma vs kappa
background = [
    NoiseTrader('nt1', seed=10),
    NoiseTrader('nt2', seed=11),
    InformedTrader('inf1', seed=20),
]

gammas = [0.05, 0.10, 0.20, 0.40]
kappas = [0.5,  1.0,  1.5,  3.0]

results = []
for g in gammas:
    for k in kappas:
        agents_sweep = background + [
            AvellanedaStoikov('as_test', seed=50, gamma=g, kappa=k,
                              size=15, horizon=500)
        ]
        market_sweep = Market(agents_sweep, n_ticks=500, seed=42)
        market_sweep.run()
        lb = {aid: pnl for aid, pnl, _ in market_sweep.leaderboard()}
        results.append({'gamma': g, 'kappa': k, 'pnl': lb.get('as_test', 0)})

sweep_df = pd.DataFrame(results)
pivot = sweep_df.pivot(index='gamma', columns='kappa', values='pnl')
pivot.index.name = 'gamma \ kappa'
print(pivot.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'κ={k}' for k in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'γ={g}' for g in pivot.index])
plt.colorbar(im, ax=ax, label='Final PnL ($)')
ax.set_title('A-S Parameter Sweep: Final PnL ($)')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.0f}', ha='center', va='center',
                fontsize=9, color='black')
plt.tight_layout()
plt.show()

---
## Part 3: TWAP — The Execution Algorithm

Market makers *provide* liquidity.  Execution algorithms *consume* it efficiently.

**TWAP (Time-Weighted Average Price):** splits a large order into equal child orders spread over a time window.  Goal: minimize market impact while completing the full order.

**Why not just send one big market order?**  A single large order walks the book — it eats through all the liquidity at each price level, pushing price against you.  TWAP disguises the trade and averages the impact.

**Reference:** Almgren & Chriss (2001). *Optimal Execution of Portfolio Transactions.* Journal of Risk.

In [ ]:
# Compare: TWAP vs. immediate market order
# We approximate an immediate order by setting duration=1

print("=== TWAP: 200 shares over 100 ticks ===")
agents_twap = [
    NoiseTrader('nt1', seed=10),
    NoiseTrader('nt2', seed=11),
    NoiseTrader('nt3', seed=12),
    NaiveMarketMaker('mm1', seed=40),
    AvellanedaStoikov('mm2', seed=50, size=15),
    TWAPAgent('twap', seed=60, target_qty=200, duration=100, use_limit=False),
]
pnl_twap, inv_twap, mkt_twap, _ = run_sim(agents_twap, n_ticks=300, seed=42)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

# Mid price — look for impact
if 'best_bid' in mkt_twap.columns and 'best_ask' in mkt_twap.columns:
    mid = (mkt_twap['best_bid'].fillna(method='ffill') +
           mkt_twap['best_ask'].fillna(method='ffill')) / 2
    axes[0].plot(mid / 100, color='steelblue', lw=1)
    axes[0].axvline(99, color='grey', lw=0.8, linestyle='--', label='TWAP window')
    axes[0].axvline(0,  color='grey', lw=0.8, linestyle='--')
    axes[0].set_title('Mid Price During TWAP Execution')
    axes[0].set_ylabel('Price ($)')
    axes[0].legend()

# TWAP inventory (should ramp from 0 → 200, then flat)
axes[1].plot(inv_twap['twap'], color='green', lw=1.5)
axes[1].axhline(200, color='grey', lw=0.8, linestyle='--', label='target=200')
axes[1].set_title('TWAP Agent Inventory (cumulative fills)')
axes[1].set_ylabel('Shares held')
axes[1].legend()

# PnL comparison: TWAP vs MMs
for col, color in [('twap', 'green'), ('mm1', 'coral'), ('mm2', 'steelblue')]:
    if col in pnl_twap:
        axes[2].plot(pnl_twap[col], label=col, color=color, lw=1.2)
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_title('PnL: TWAP vs Market Makers')
axes[2].set_ylabel('PnL ($)')
axes[2].legend()

plt.tight_layout()
plt.show()

twap_final_inv = inv_twap['twap'].iloc[-1] if 'twap' in inv_twap else '?'
print(f"TWAP final inventory: {twap_final_inv} shares (target: 200)")

---
## Part 4: Mean Reversion vs. Momentum

Two directional strategies with opposite hypotheses:

| Strategy | Hypothesis | Wins when |
|---|---|---|
| `MomentumTrader` | Trends persist | Fundamental value drifts; informed flow drives sustained moves |
| `MeanReversionAgent` | Prices revert to fair value | Noise dominates; temporary order imbalances correct quickly |

In a market with a **slow-moving fundamental** (small drift), noise dominates and mean reversion wins.  In a market with a **fast-moving fundamental** (large drift), price dislocations are real signals and momentum wins.

Let's test both regimes.

In [ ]:
base_pop = [
    NoiseTrader('nt1', seed=10),
    NoiseTrader('nt2', seed=11),
    InformedTrader('inf1', seed=20),
    NaiveMarketMaker('mm1', seed=40),
]

# Low-drift fundamental: noise dominates → mean reversion should win
print("=== Low-drift fundamental ===")
low_drift_agents = base_pop + [
    MomentumTrader('mom', seed=30, lookback=15, threshold_bps=20),
    MeanReversionAgent('mr',  seed=60, lookback=20, entry_bps=15),
]
pnl_low, inv_low, mkt_low, _ = run_sim(
    low_drift_agents, n_ticks=1000, seed=42,
    fundamental_kwargs={'drift': 0.0, 'vol_bps': 8.0}  # no drift, default vol
)

In [ ]:
# High-drift fundamental: trends → momentum should win
print("=== High-drift fundamental ===")
high_drift_agents = base_pop + [
    MomentumTrader('mom', seed=30, lookback=15, threshold_bps=20),
    MeanReversionAgent('mr',  seed=60, lookback=20, entry_bps=15),
]
pnl_high, inv_high, mkt_high, _ = run_sim(
    high_drift_agents, n_ticks=1000, seed=42,
    fundamental_kwargs={'drift': 0.0001, 'vol_bps': 8.0}  # positive drift → trending market
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col_idx, (pnl_df, mkt_df, title) in enumerate([
    (pnl_low,  mkt_low,  'Low Drift: Mean Reversion regime'),
    (pnl_high, mkt_high, 'High Drift: Momentum regime'),
]):
    ax_pnl = axes[0, col_idx]
    ax_mid = axes[1, col_idx]

    if 'mom' in pnl_df:
        ax_pnl.plot(pnl_df['mom'], label='Momentum',      color='orange',    lw=1.5)
    if 'mr' in pnl_df:
        ax_pnl.plot(pnl_df['mr'],  label='MeanReversion', color='steelblue', lw=1.5)
    ax_pnl.axhline(0, color='black', lw=0.8)
    ax_pnl.set_title(title)
    ax_pnl.set_ylabel('PnL ($)')
    ax_pnl.legend()

    if 'fundamental' in mkt_df.columns:
        ax_mid.plot(mkt_df['fundamental'] / 100, color='grey', lw=1, label='fundamental')
    if 'last_price' in mkt_df.columns:
        ax_mid.plot(mkt_df['last_price'].fillna(method='ffill') / 100,
                    color='steelblue', lw=0.8, alpha=0.7, label='last price')
    ax_mid.set_title('Price vs Fundamental')
    ax_mid.set_ylabel('Price ($)')
    ax_mid.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Discussion questions:**

1. In the low-drift regime, which agent wins?  Does this match your intuition?
2. In the high-drift regime, does momentum benefit more from the trend itself, or from chasing the informed trader's flow?
3. What would happen if you added a second `MeanReversionAgent` to the high-drift market?  Would they both lose, or would competition reduce each other's edge?

> **Key insight:** Neither strategy has an absolute edge.  The winning hypothesis depends on market microstructure — specifically, whether price deviations from fair value are *informative* (momentum) or *temporary* (mean reversion).

---
## Part 5: The Grand Tournament

Put all agent types in a single market and let them compete for 2000 ticks.

In [ ]:
print("=== Grand Tournament: All Agents ===")
tournament_agents = [
    # Background population
    NoiseTrader('noise_1', seed=10, intensity=0.3, noise_bps=40),
    NoiseTrader('noise_2', seed=11, intensity=0.3, noise_bps=40),
    NoiseTrader('noise_3', seed=12, intensity=0.3, noise_bps=40),
    InformedTrader('informed', seed=20, edge_bps=20, size=12),

    # Market makers
    NaiveMarketMaker('naive_mm',  seed=40, half_spread=5, size=20),
    AvellanedaStoikov('as_mm',    seed=50, gamma=0.1, kappa=1.5, size=15, horizon=2000),

    # Directional
    MomentumTrader('momentum',   seed=30, lookback=20, threshold_bps=15, size=8),
    MeanReversionAgent('mean_rev', seed=60, lookback=30, entry_bps=20, size=10),
]

pnl_T, inv_T, mkt_T, mkt_obj_T = run_sim(tournament_agents, n_ticks=2000, seed=99)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

style = {
    'naive_mm'   : ('coral',     '--', 1.8),
    'as_mm'      : ('steelblue', '-',  2.0),
    'momentum'   : ('orange',    '-',  1.5),
    'mean_rev'   : ('purple',    '-',  1.5),
    'informed'   : ('green',     '-',  1.5),
    'noise_1'    : ('grey',      ':',  0.8),
    'noise_2'    : ('grey',      ':',  0.8),
    'noise_3'    : ('grey',      ':',  0.8),
}

for aid, (color, ls, lw) in style.items():
    if aid in pnl_T.columns:
        lbl = aid if not aid.startswith('noise') else ('noise traders' if aid == 'noise_1' else None)
        axes[0].plot(pnl_T[aid], color=color, linestyle=ls, linewidth=lw, label=lbl)

axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Grand Tournament: PnL over 2000 Ticks')
axes[0].set_ylabel('PnL ($)')
axes[0].legend(loc='upper left', fontsize=9)

# Inventory
for aid, (color, ls, lw) in style.items():
    if aid in inv_T.columns and not aid.startswith('noise'):
        axes[1].plot(inv_T[aid], color=color, linestyle=ls, linewidth=lw, label=aid)

axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Inventory over Time')
axes[1].set_ylabel('Shares')
axes[1].legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Compute per-agent Sharpe from daily PnL changes
daily_pnl = pnl_T.diff().dropna()
sharpe = (daily_pnl.mean() / daily_pnl.std() * np.sqrt(252)).round(3)
final_pnl = pnl_T.iloc[-1].round(2)
max_inv   = inv_T.abs().max().round(0)

summary = pd.DataFrame({
    'Final PnL ($)' : final_pnl,
    'Sharpe (ann.)' : sharpe,
    'Max |Inventory|': max_inv,
}).sort_values('Final PnL ($)', ascending=False)
summary

---
## Part 6: Build Your Own Agent

Now it's your turn.  Build an agent that outperforms the existing ones in the tournament.

**The interface:**
```python
class MyAgent(Agent):
    def on_tick(self, state: MarketState) -> list[Order]:
        ...
```

**What `state` gives you:**

| Attribute | Type | Description |
|---|---|---|
| `state.tick` | `int` | Current tick number |
| `state.best_bid` | `int \| None` | Best bid price (cents) |
| `state.best_ask` | `int \| None` | Best ask price (cents) |
| `state.mid` | `float \| None` | `(bid + ask) / 2` |
| `state.last_price` | `int \| None` | Last trade price |
| `state.depth` | `dict` | Full book: `{"bids": [(price, qty)...], "asks": [...]}` |
| `state.recent_trades` | `list` | Trades from the previous tick |
| `state.position` | `int` | Your signed inventory |
| `state.cash` | `int` | Your cash in cents |
| `state.my_open_orders` | `list` | Your resting orders |

**Building orders:**
```python
self.limit(side, price, qty)   # resting limit order
self.market(side, qty)         # immediate market order
self.cancel(order_id)          # cancel a resting order
```

In [ ]:
class MyAgent(Agent):
    """
    Your agent here.  The goal: outperform NaiveMarketMaker and
    AvellanedaStoikov in the tournament below.
    
    Ideas to consider:
    - Combine inventory management (A-S) with a directional signal (momentum or MR)
    - Use order book depth to detect informed flow and pause quoting
    - Implement a dynamic spread that widens during high-volatility periods
    - Add a circuit breaker that goes flat when the spread disappears
    """

    def __init__(self, agent_id, seed=0):
        super().__init__(agent_id, seed)
        self._history = []

    def on_tick(self, state: MarketState) -> list[Order]:
        if state.mid is None:
            return []

        self._history.append(state.mid)

        # --- YOUR STRATEGY HERE ---
        # Example: simple fixed spread around mid
        half_spread = 8
        bid = round(state.mid) - half_spread
        ask = round(state.mid) + half_spread

        orders = [self.cancel(oid) for oid, *_ in state.my_open_orders]
        if state.position < 200:
            orders.append(self.limit(Side.BUY, max(1, bid), 15))
        if state.position > -200:
            orders.append(self.limit(Side.SELL, ask, 15))
        return orders

# Test your agent in the tournament
print("=== Custom Agent Tournament ===")
custom_agents = [
    NoiseTrader('noise_1', seed=10),
    NoiseTrader('noise_2', seed=11),
    NoiseTrader('noise_3', seed=12),
    InformedTrader('informed', seed=20),
    MomentumTrader('momentum', seed=30),
    NaiveMarketMaker('naive_mm', seed=40),
    AvellanedaStoikov('as_mm', seed=50, gamma=0.1, kappa=1.5, horizon=2000),
    MyAgent('my_agent', seed=99),
]
pnl_my, inv_my, mkt_my, _ = run_sim(custom_agents, n_ticks=2000, seed=99)

---
## Part 7: Challenges

Complete any **two** of the following.

### Challenge A: The Volatility Shock

Use `market.at_tick()` to inject a volatility shock at tick 500 (double the noise traders' noise_bps for 100 ticks).  Compare how `NaiveMarketMaker` and `AvellanedaStoikov` respond to the shock in terms of:
- Spread widening (A-S should widen automatically; naive MM won't)
- Inventory management during the shock
- PnL stability before vs. after

In [ ]:
# Challenge A — your solution here
nt_shock1 = NoiseTrader('nt1', seed=10)
nt_shock2 = NoiseTrader('nt2', seed=11)
shock_agents = [
    nt_shock1, nt_shock2,
    InformedTrader('inf1', seed=20),
    NaiveMarketMaker('naive_mm', seed=40),
    AvellanedaStoikov('as_mm', seed=50, gamma=0.1, kappa=1.5, horizon=1000),
]

def inject_shock(m):
    for a in [nt_shock1, nt_shock2]:
        a.noise_bps *= 2

def remove_shock(m):
    for a in [nt_shock1, nt_shock2]:
        a.noise_bps //= 2

market_shock = Market(shock_agents, n_ticks=1000, seed=42)
market_shock.at_tick(500, inject_shock)
market_shock.at_tick(600, remove_shock)

telem_shock = {'naive_mm': [], 'as_mm': []}
def collect_shock(m):
    mark = m.engine.last_price or m.fundamental.value
    if mark:
        for aid in telem_shock:
            acct = m.accounts.get(aid)
            if acct:
                telem_shock[aid].append(acct.value(mark) / 100)

for t in range(1, 1001):
    market_shock.at_tick(t, collect_shock)
market_shock.run()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(telem_shock['naive_mm'], label='NaiveMarketMaker', color='coral')
ax.plot(telem_shock['as_mm'],   label='AvellanedaStoikov', color='steelblue')
ax.axvspan(499, 599, alpha=0.15, color='red', label='shock window')
ax.axhline(0, color='black', lw=0.6)
ax.set_title('PnL During Volatility Shock (ticks 500–600)')
ax.set_ylabel('PnL ($)')
ax.legend()
plt.tight_layout()
plt.show()

print('Final PnL:')
for aid, vals in telem_shock.items():
    print(f'  {aid}: ${vals[-1]:.2f}')

### Challenge B: Optimal TWAP Duration

TWAP faces a fundamental tradeoff:
- **Short duration** → completes quickly, but each child order is large → high market impact
- **Long duration** → minimal impact, but exposes you to adverse price movement for longer

Run `TWAPAgent` with durations `[10, 25, 50, 100, 200]` ticks and a fixed target of 200 shares.  Plot the **execution VWAP** (volume-weighted average price of all fills) against the market VWAP for each duration.  What is the optimal duration in this microstructure?

In [ ]:
# Challenge B — your solution here
durations = [10, 25, 50, 100, 200]
twap_results = []

for dur in durations:
    bg = [
        NoiseTrader('nt1', seed=10),
        NoiseTrader('nt2', seed=11),
        NaiveMarketMaker('mm1', seed=40),
    ]
    twap_agent = TWAPAgent('twap', seed=60, target_qty=200, duration=dur, use_limit=False)
    market_t = Market(bg + [twap_agent], n_ticks=max(dur + 100, 300), seed=42)
    market_t.run()
    lb = {aid: pnl for aid, pnl, _ in market_t.leaderboard()}
    twap_results.append({'duration': dur, 'final_pnl': lb.get('twap', 0)})

print(pd.DataFrame(twap_results).set_index('duration'))

### Challenge C: Deploy to the Live Arena

Once you've developed and refined your agent above, deploy it to the class Arena server.  Use the `RemoteAgent` class:

```python
from convexpi.arena import RemoteAgent

class MyRemoteAgent(RemoteAgent):
    def on_tick(self, state: MarketState) -> list[Order]:
        # your on_tick logic here (same as above)
        ...

# Connect and run
MyRemoteAgent(
    agent_id='your_name',
    ws_url='ws://arena.convexitylabs.io:8765',
).start()
```

The live Arena uses a real-time tick interval and a real leaderboard.  Your in-process strategy should work without modification.

---
## Wrap-up

**What you covered:**

| Concept | Where |
|---|---|
| Adverse selection cost decomposition | Part 1 |
| Avellaneda-Stoikov reservation price + optimal spread | Part 2 |
| A-S parameter tuning (gamma, kappa) | Part 2 |
| TWAP execution — splitting large orders | Part 3 |
| Mean reversion vs. momentum: regime dependence | Part 4 |
| Grand tournament: all agent types compete | Part 5 |
| Building and testing a custom agent | Part 6 |

**Key takeaways:**

1. **Adverse selection is the dominant risk for market makers.** The naive MM is profitable against noise but loses severely to informed flow.  Inventory-aware quoting (A-S) helps but doesn't eliminate this exposure.
2. **Optimal quoting is regime-dependent.** A-S performs better when the fundamental moves slowly and inventory accumulates gradually.  In a fast-moving market, neither MM strategy is particularly safe.
3. **Execution algorithms solve a different problem.** TWAP, VWAP, and Almgren-Chriss don't maximize PnL — they minimize implementation shortfall for a pre-committed trade.
4. **No agent has unconditional alpha.** Momentum wins in trending markets; mean reversion wins in noisy markets.  Knowing your regime is as important as knowing your strategy.
5. **In-process code transfers to the live Arena directly.** The `on_tick` interface is the same whether you're running locally or competing on the class leaderboard.